# Shadow casting by a cold sphere: analytical solution

Stationary, monochromatic solution of the absorption–emission problem used as the reference for the EMU shadow test. A homogeneous emitting sphere sits in vacuum. A smaller, cold, purely absorbing sphere lies downstream of it. Neutrinos are massless, spacetime is flat, and there is no scattering, fluid motion, or flavor evolution.

The derivation below follows the benchmark writeup. The implementation uses the EMU run's geometry, opacities, and energy bin. The angular mesh is a θ–φ product grid defined in this notebook, so the result does not depend on the simulation's direction count. Where the writeup and EMU disagree, the difference is stated and the EMU convention for \(f\), \(n\), and \(\mathbf{F}\) is the one evaluated.


## Assumptions

**Geometry.** The emitter is the ball \(\mathcal{E}=\{|\mathbf{x}-\mathbf{x}_e|\le R_e\}\) and the absorber is \(\mathcal{A}=\{|\mathbf{x}-\mathbf{x}_a|\le R_a\}\), with

$$
R_e>R_a>0,\qquad d=|\mathbf{x}_a-\mathbf{x}_e|>R_e+R_a.
$$

The bodies are static, homogeneous, and disjoint. In the writeup the absorber sits at \(d\hat{\mathbf{z}}\). In this run the centers differ only in \(x\), so the separation axis is

$$
\hat{\mathbf{e}}_{\mathrm{sep}}=(\mathbf{x}_a-\mathbf{x}_e)/d=\hat{\mathbf{x}}.
$$

Every formula in the writeup is used with \(\hat{\mathbf{z}}\to\hat{\mathbf{e}}_{\mathrm{sep}}\) and \(n_z\to \hat{\mathbf{n}}\cdot\hat{\mathbf{e}}_{\mathrm{sep}}\).

**Transport.** One neutrino species, one energy \(E\), no incident radiation from infinity. The stationary Boltzmann equation on straight rays reduces to

$$
\hat{\mathbf{n}}\cdot\nabla f=\kappa(\mathbf{x})\,[S(\mathbf{x})-f].
$$

Here \(f\) is the occupation number, \(\kappa\) is the absorption coefficient (inverse length), and \(S\) is the dimensionless source function. With the collision operator written \(\kappa(S-f)\),

$$
S_e=\left[1+\exp\left(\frac{E-\mu_e}{T_e}\right)\right]^{-1}
$$

inside the emitter. The absorber is cold: its emissivity is set to zero (\(S_a=0\)), so \(\kappa_a\) only removes occupation. Outside \(\mathcal{E}\cup\mathcal{A}\) the benchmark takes \(\kappa=0\).

**Propagation direction.** \(\hat{\mathbf{n}}\) is the direction of travel, the same vector EMU stores as \(\hat{\mathbf{p}}=\mathbf{p}/E\). The backward ray used for the formal solution is

$$
\mathbf{y}(s)=\mathbf{x}-s\hat{\mathbf{n}},\qquad s\ge 0.
$$


## Where the writeup and EMU differ

| Quantity | Writeup | This notebook |
|---|---|---|
| Separation axis | \(+\hat{\mathbf{z}}\) | \(+\hat{\mathbf{x}}\), the axis of the Cartesian run |
| \(f\) | Occupation number | Same occupation number. EMU's particle attribute `N00_Re` is not \(f\) |
| Moments | \(J=\oint f\,d\Omega/4\pi\), \(\mathbf{H}=\oint f\hat{\mathbf{n}}\,d\Omega/4\pi\) | Number density \(n\) and number flux \(\mathbf{F}\) in \(\mathrm{cm}^{-3}\), using EMU's phase-space volume |
| Flux factor of \(c\) | \(\mathbf{H}\) is dimensionless like \(f\) | EMU mesh `Fx00_Re` stores \(\mathbf{F}\), which is the number flux divided by \(c\). No extra factor of \(c\) is applied |
| Vacuum opacity | \(\kappa=0\) | The run assigns \(\kappa_{\mathrm{vac}}\sim 10^{-9}\,\mathrm{cm}^{-1}\). The solution uses \(\kappa=0\) outside both spheres, as in the benchmark |
| Stimulated absorption | \(\kappa=\chi/(1-f^{\mathrm{eq}})\) | `Do_Pauli_blocking = 0`, so the NuLib \(\kappa\) is used as stored. \(S_e\sim 3\times 10^{-13}\), so \(\kappa\) and \(\kappa/(1-S_e)\) differ by a negligible amount |
| Absorber emission | \(S_a=0\) | Same. The absorber's Fermi–Dirac occupation underflows to 0 at this energy |
| Boundary | No radiation incident from infinity | `periodic_empty` zeros the number content of particles that wrap, so wrapped particles re-enter empty |

Particle number content and occupation are related by the phase-space volume of that macroparticle,

$$
N_{00}=f\,\frac{V_{\mathrm{phase}}}{(2\pi\hbar c)^3},\qquad
f=N_{00}\,\frac{(2\pi\hbar c)^3}{V_{\mathrm{phase}}}.
$$

`particle_input.dat` stores \(V_{\mathrm{phase}}\) without the cell volume. After initialization EMU multiplies both \(N\) and \(V_{\mathrm{phase}}\) by \(V_{\mathrm{cell}}/n_{\mathrm{loc}}\). Plotfiles therefore contain \(V_{\mathrm{phase}}^{\mathrm{plot}}=V_{\mathrm{phase}}\,V_{\mathrm{cell}}\). Deposition divides by \(V_{\mathrm{cell}}\), so the mesh number density is

$$
n=\sum_a f_a\,\frac{V_{\mathrm{phase},a}}{(2\pi\hbar c)^3},
$$

with the **unscaled** \(V_{\mathrm{phase}}\). The same weight with an extra \(\hat{\mathbf{n}}_a\) gives \(\mathbf{F}\). Units of both are \(\mathrm{cm}^{-3}\).


## Ray solution

Along the backward ray, \(df/ds=\kappa(f-S)\). With nothing incident from infinity the formal solution collapses, segment by segment, to

$$
f(\mathbf{x},\hat{\mathbf{n}})=S_e\left(1-e^{-\kappa_e \ell_e}\right)\exp\!\left[-\kappa_a \ell_a\,\Theta(\hat{\mathbf{n}}\cdot\hat{\mathbf{e}}_{\mathrm{sep}})\right].
$$

\(\ell_e\) and \(\ell_a\) are the lengths of the backward ray inside each ball. \(\Theta\) is the Heaviside function. It is required because the absorber does not emit or scatter: absorber material on the far side of the emitter does not change \(f\).

For a ball of radius \(R\) centered at \(\mathbf{x}_c\), let \(\mathbf{r}=\mathbf{x}-\mathbf{x}_c\), \(D=|\mathbf{r}|\), and \(b=\mathbf{r}\cdot\hat{\mathbf{n}}\). The intersections \(\mathbf{y}(s)\) lie at

$$
s_{\pm}=b\pm\sqrt{\Delta},\qquad \Delta=R^2-D^2+b^2.
$$

The backward half-ray meets the ball only if \(\Delta>0\) and \(s_+>0\). The clipped chord is

$$
\ell=s_+-\max(s_-,0),
$$

and \(\ell=0\) otherwise. This one expression covers an exterior observer, an interior observer (\(\ell=s_+\)), a miss, and a tangent ray (\(\Delta=0\Rightarrow\ell=0\)). Grazing rays are a set of measure zero. At finite opacity \(f\) is continuous across them because the chord vanishes.

The two balls are disjoint, so their intervals on any line are disjoint. The absorber interval lies closer to \(\mathbf{x}\) than the emitter interval if and only if \(\hat{\mathbf{n}}\cdot\hat{\mathbf{e}}_{\mathrm{sep}}>0\). Clipping of downstream material is already contained in \(\ell_a\). The implementation takes \(\Theta(0)=0\); that plane has measure zero.

Consequences used as checks:

- A ray that misses the emitter has \(f=0\).
- Inside the emitter the field is the isolated-sphere solution. A foreground absorber cannot sit at \(s<0\).
- On the emitter side of the internal center of similitude, and everywhere in the vacuum gap, \(f\) equals the isolated-sphere value.
- In the opaque limit the absorber is a visibility mask on directions. This run has \(\tau_a=\kappa_a R_a\approx 48\), so the umbra is dark only up to a very small transmitted occupation, not exactly zero.


## Shadow surfaces

Place the axial coordinate \(s=(\mathbf{x}-\mathbf{x}_e)\cdot\hat{\mathbf{e}}_{\mathrm{sep}}\) at the emitter center and the cylindrical radius \(\varpi\) about that axis. The internal and external centers of similitude, and the angles of the common tangents, are

$$
s_i=\frac{d R_e}{R_e+R_a},\qquad
s_x=\frac{d R_e}{R_e-R_a},\qquad
\sin\beta_u=\frac{R_e-R_a}{d},\qquad
\sin\beta_p=\frac{R_e+R_a}{d}.
$$

\(\beta_u\) is the external (umbra) cone and \(\beta_p\) is the internal (penumbra) cone. Revolving the downstream tangents gives

$$
\begin{aligned}
\varpi_u(s)&=(s_x-s)\tan\beta_u,\\
\varpi_u'(s)&=(s-s_x)\tan\beta_u,\\
\varpi_p(s)&=(s-s_i)\tan\beta_p.
\end{aligned}
$$

Outside the absorber, the umbra is \(\varpi<\varpi_u(s)\) for \(d+R_a\sin\beta_u\le s\le s_x\), the antumbra is \(\varpi<\varpi_u'(s)\) for \(s>s_x\), and the penumbra is the rest of \(\varpi<\varpi_p(s)\) for \(s\ge d-R_a\sin\beta_p\). Because \(R_e>R_a\) the umbra closes at a finite tip \(s_x\). These surfaces classify points. The occupation itself is always computed from the chord formula, including at finite \(\kappa_a\).


## Discrete moments, sampling, and memory

Directions are a product mesh in polar angle \(\theta\) and azimuth \(\phi\), with \(N_\theta\) and \(N_\phi\) bins. \(\theta=0\) is \(+\hat{\mathbf{z}}\) and \(\phi=0\) is \(+\hat{\mathbf{x}}\). Bin \((i,j)\) covers

$$
\theta\in[i,\,i+1]\,\frac{\pi}{N_\theta},\qquad
\phi\in[j,\,j+1]\,\frac{2\pi}{N_\phi},
$$

and the direction \(\hat{\mathbf{n}}_a\) is the center of that bin,

$$
\hat{\mathbf{n}}=(\sin\theta\cos\phi,\,\sin\theta\sin\phi,\,\cos\theta).
$$

The weight is the exact solid angle of the bin, \(\Delta\Omega_a=(\cos\theta_i-\cos\theta_{i+1})\,\Delta\phi\), so \(\sum_a\Delta\Omega_a=4\pi\) for any \(N_\theta\) and \(N_\phi\). The normalized weights \(w_a=\Delta\Omega_a/(4\pi)\) satisfy \(\sum_a w_a=1\). They are not equal: bins near the poles carry less solid angle. The flat index is \(a=i N_\phi+j\).

The unscaled phase-space volume of each bin, for one energy group, is

$$
V_{\mathrm{phase},a}=\Delta\Omega_a\,\frac{E_{\mathrm{hi}}^3-E_{\mathrm{lo}}^3}{3},
$$

with \(E\) in erg. Equal-weight EMU quadrature is the special case \(\Delta\Omega_a=4\pi/N_{\mathrm{dir}}\). The cell-center moments keep the same sums,

$$
\begin{aligned}
n(\mathbf{x})&=\sum_a f(\mathbf{x},\hat{\mathbf{n}}_a)\,\frac{V_{\mathrm{phase},a}}{(2\pi\hbar c)^3},\\[4pt]
\mathbf{F}(\mathbf{x})&=\sum_a f(\mathbf{x},\hat{\mathbf{n}}_a)\,\hat{\mathbf{n}}_a\,\frac{V_{\mathrm{phase},a}}{(2\pi\hbar c)^3}.
\end{aligned}
$$

\(n\) matches mesh `N00_Re(1|ccm)` before shape-function deposition. \(\mathbf{F}\) matches `Fx00_Re`, `Fy00_Re`, `Fz00_Re` in the same sense: it is \(\langle\hat{\mathbf{n}}\rangle n\), not \(c\) times that. Because this \(\{\hat{\mathbf{n}}_a\}\) is not a simulation direction set, \(n\) and \(\mathbf{F}\) are the analytic field on the notebook mesh.

**Spatial sampling.** `f[i,j,k,a]`, `n[i,j,k]`, and `F[i,j,k,:]` are values at the **cell center**

$$
\mathbf{x}_{ijk}=\big((i+\tfrac12)\Delta x,\ (j+\tfrac12)\Delta y,\ (k+\tfrac12)\Delta z\big),
$$

with \(i,j,k\) zero-based. That is the initial location of the single macroparticle per direction per cell (`nppc = (1,1,1)`). It is not a cell average. The mesh fields in the plotfile are additionally smoothed by the deposition shape function (`deposit_method = 1`). Comparing those mesh fields to this \(n\) and \(\mathbf{F}\) therefore still includes the shape-function kernel. A cell average of the exact solution would be a separate tensor-product quadrature of \(f\) over the cell, with breakpoints on the shadow surfaces; that average is not what is stored here.

**Memory.** The full array `f[i,j,k,a]` in float64 has size `ncell[0]*ncell[1]*ncell[2]*N_theta*N_phi*8` bytes. It is not allocated. `f_at_cell(i,j,k)` returns the occupations at one center. `evaluate_chunk` evaluates a rectangular index slab and refuses to return an `f` array larger than 2 GiB; `store_f=False` returns only \(n\) and \(\mathbf{F}\), which are a few tens of megabytes for the whole mesh.


In [ ]:
import numpy as np


In [ ]:
# Grid and material parameters of the Cartesian shadow run.
# Lengths are cm, temperatures and chemical potentials are MeV, opacities are 1/cm.
ncell = (192, 64, 64)
Lx = 8424000.0  # (R + 0.2 R) * 6, R = 1.17e6 cm
Ly = 2808000.0  # (R + 0.2 R) * 2
Lz = 2808000.0

dx = Lx / ncell[0]
dy = Ly / ncell[1]
dz = Lz / ncell[2]
cell_volume = dx * dy * dz

R = 1170000.0  # cm
R_e = R
R_a = 1.5 * R / 4.5  # R/3

r_emitter_sphere_center_cm = (1425937.5, 1425937.5, 1425937.5)
r_shadowing_sphere_center_cm = (4935937.5, 1425937.5, 1425937.5)
center_e = np.array(r_emitter_sphere_center_cm, dtype=np.float64)
center_a = np.array(r_shadowing_sphere_center_cm, dtype=np.float64)

mu_e = -3.130432908491162  # MeV
T_e = 5.1189  # MeV
kappa_e = 0.00000854544806011998  # 1/cm, NuLib absorption, used as stored

mu_a = -19.95627340541274  # MeV
T_a = 0.0566631  # MeV
kappa_a = 0.00012391882339512724  # 1/cm

mu_vacuum = -25.859107462353197  # MeV
T_vacuum = 0.05666306  # MeV
kappa_vacuum = 1.4877457720665111e-9  # 1/cm; recorded, not used in f

# NuLib group containing the run's monochromatic energy (st_radiating_sphere.py).
E_MeV = 144.74
E_lo_MeV = 128.65
E_hi_MeV = 160.83

# EMU CGS constants (Source/Constants.H and amrex_plot_tools.py).
eV = 1.60218e-12  # erg
c = 2.99792458e10  # cm/s
hbar = 1.05457266e-27  # erg s
hbar_c = hbar * c  # erg cm

E_erg = E_MeV * 1e6 * eV
E_lo_erg = E_lo_MeV * 1e6 * eV
E_hi_erg = E_hi_MeV * 1e6 * eV


def fermi_dirac_occupation(energy_MeV, mu_MeV, temperature_MeV):
    """Occupation [exp((E-mu)/T)+1]^(-1), stable for a cold absorber."""
    exponent = (energy_MeV - mu_MeV) / temperature_MeV
    if exponent >= 700.0:
        return 0.0
    if exponent > 60.0:
        return float(np.exp(-exponent))
    return float(1.0 / (np.exp(exponent) + 1.0))


S_e = fermi_dirac_occupation(E_MeV, mu_e, T_e)
S_a_equilibrium = fermi_dirac_occupation(E_MeV, mu_a, T_a)
S_vacuum_equilibrium = fermi_dirac_occupation(E_MeV, mu_vacuum, T_vacuum)

separation = center_a - center_e
d = float(np.linalg.norm(separation))
e_sep = separation / d

tau_e = kappa_e * R_e
tau_a = kappa_a * R_a
sin_beta_u = (R_e - R_a) / d
sin_beta_p = (R_e + R_a) / d
beta_u = float(np.arcsin(sin_beta_u))
beta_p = float(np.arcsin(sin_beta_p))
# Axial coordinates measured from the emitter center, along e_sep.
x_internal = d * R_e / (R_e + R_a)  # internal center of similitude
x_external = d * R_e / (R_e - R_a)  # umbral tip

if not (R_e > R_a > 0.0 and d > R_e + R_a):
    raise ValueError("emitter and absorber are not a disjoint shadow pair")

print(f"cell size dx, dy, dz = {dx:.6g}, {dy:.6g}, {dz:.6g} cm")
print(f"R_e = {R_e:.6g} cm, R_a = {R_a:.6g} cm, d = {d:.6g} cm")
print(f"R_a/R_e = {R_a / R_e:.6g}, d/R_e = {d / R_e:.6g}")
print(f"tau_e = {tau_e:.6g}, tau_a = {tau_a:.6g}")
print(f"gap / R_e = {(d - R_e - R_a) / R_e:.6g}")
print(f"S_e = {S_e:.6e}")
print(f"absorber equilibrium occupation (unused; source is 0) = {S_a_equilibrium:.3e}")
print(f"vacuum equilibrium occupation (unused) = {S_vacuum_equilibrium:.3e}")
print(f"vacuum optical depth across Lx if kappa_vacuum were kept = {kappa_vacuum * Lx:.4e}")
print(f"s_i = {x_internal:.6g} cm, s_tip = {x_external:.6g} cm")
print(f"beta_u = {np.degrees(beta_u):.4f} deg, beta_p = {np.degrees(beta_p):.4f} deg")
print(f"e_sep = {e_sep}")


In [ ]:
# Angular mesh. Change n_theta and n_phi to refine the quadrature.
# This mesh replaces the simulation direction set.
n_theta = 32  # bins in polar angle, theta in [0, pi]
n_phi = 64    # bins in azimuth, phi in [0, 2 pi]

theta_edges = np.linspace(0.0, np.pi, n_theta + 1)
phi_edges = np.linspace(0.0, 2.0 * np.pi, n_phi + 1)
theta_centers = 0.5 * (theta_edges[:-1] + theta_edges[1:])
phi_centers = 0.5 * (phi_edges[:-1] + phi_edges[1:])
delta_phi = np.diff(phi_edges)
# Integral of sin(theta) over a theta bin. Sums to 2 over all bins.
delta_cos = np.cos(theta_edges[:-1]) - np.cos(theta_edges[1:])

theta_grid, phi_grid = np.meshgrid(theta_centers, phi_centers, indexing="ij")
dcos_grid, dphi_grid = np.meshgrid(delta_cos, delta_phi, indexing="ij")
# Flat index a = i_theta * n_phi + i_phi.
theta_a = theta_grid.ravel()
phi_a = phi_grid.ravel()
solid_angle = (dcos_grid * dphi_grid).ravel()

phat = np.column_stack(
    (
        np.sin(theta_a) * np.cos(phi_a),
        np.sin(theta_a) * np.sin(phi_a),
        np.cos(theta_a),
    )
)
n_directions = phat.shape[0]
weights = solid_angle / (4.0 * np.pi)
# Same energy-bin factor as EMU, with this mesh's solid angle instead of 4*pi/Ndir.
energy_cube = (E_hi_erg**3 - E_lo_erg**3) / 3.0
vphase_a = solid_angle * energy_cube

if n_directions != n_theta * n_phi:
    raise RuntimeError("direction count does not equal n_theta * n_phi")
if abs(float(np.sum(solid_angle)) - 4.0 * np.pi) > 1e-10:
    raise RuntimeError("bin solid angles do not sum to 4*pi")
if abs(float(np.sum(weights)) - 1.0) > 1e-12:
    raise RuntimeError("angular weights do not sum to 1")
norm = np.linalg.norm(phat, axis=1)
if np.max(np.abs(norm - 1.0)) > 1e-12:
    raise RuntimeError("mesh directions are not unit vectors")

full_f_bytes = int(np.prod(ncell)) * n_directions * np.dtype(np.float64).itemsize
n_bytes = int(np.prod(ncell)) * np.dtype(np.float64).itemsize
flux_bytes = n_bytes * 3

print(f"angular mesh: n_theta = {n_theta}, n_phi = {n_phi}, n_directions = {n_directions}")
print("theta = 0 at +z, phi = 0 at +x; direction index a = i_theta * n_phi + i_phi")
print(f"sum DeltaOmega = {solid_angle.sum():.16f} (4*pi = {4.0 * np.pi:.16f})")
print(f"sum w_a = {weights.sum():.16f}")
print(f"DeltaOmega min/max = {solid_angle.min():.6e} / {solid_angle.max():.6e} sr")
print(f"sum Vphase = {vphase_a.sum():.6e} erg^3")
print(f"full f[i,j,k,a] float64 = {full_f_bytes / 1024**3:.2f} GiB; not allocated")
print(f"full n and F float64 = {(n_bytes + flux_bytes) / 1024**2:.1f} MiB")


In [ ]:
def clipped_chord(position, direction, center, radius):
    """Length of the backward half-ray inside a ball.

    Backward ray: y(s) = position - s * direction, s >= 0.
    `position` and `direction` broadcast on every axis except the last,
    which holds Cartesian components. A miss, a tangent ray (Delta = 0),
    and a sphere that lies only downstream (s_+ <= 0) all return 0.
    An interior point returns s_+.
    """
    relative = np.asarray(position, dtype=np.float64) - np.asarray(center, dtype=np.float64)
    direction = np.asarray(direction, dtype=np.float64)
    impact_projection = np.sum(relative * direction, axis=-1)
    distance_squared = np.sum(relative * relative, axis=-1)
    delta = radius * radius - distance_squared + impact_projection * impact_projection
    root = np.sqrt(np.maximum(delta, 0.0))
    s_plus = impact_projection + root
    s_minus = impact_projection - root
    length = s_plus - np.maximum(s_minus, 0.0)
    hits = (delta > 0.0) & (s_plus > 0.0)
    return np.where(hits, length, 0.0)


def f_on_points(points, directions, kappa_absorber=None, max_bytes=256 * 1024**2):
    """Occupation f at `points` (N, 3) for `directions` (A, 3).

    Returns an array of shape (N, A). Large point sets are evaluated in
    batches so the temporary (N, A) arrays stay near `max_bytes`.
    `kappa_absorber=0` is the isolated emitter.
    """
    if kappa_absorber is None:
        kappa_absorber = kappa_a
    points = np.asarray(points, dtype=np.float64)
    directions = np.asarray(directions, dtype=np.float64)
    if points.ndim == 1:
        points = points.reshape(1, 3)
    if directions.ndim == 1:
        directions = directions.reshape(1, 3)
    n_points = points.shape[0]
    n_dir = directions.shape[0]
    if n_points > 1 and n_points * n_dir * 8 > max_bytes:
        batch = max(1, int(max_bytes // (n_dir * 8)))
        values = np.empty((n_points, n_dir), dtype=np.float64)
        for start in range(0, n_points, batch):
            stop = min(start + batch, n_points)
            values[start:stop] = f_on_points(
                points[start:stop],
                directions,
                kappa_absorber=kappa_absorber,
                max_bytes=np.inf,
            )
        return values

    ell_emitter = clipped_chord(points[:, None, :], directions[None, :, :], center_e, R_e)
    ell_absorber = clipped_chord(points[:, None, :], directions[None, :, :], center_a, R_a)
    # Foreground only when the absorber center lies upstream of the emitter
    # center along the backward ray. Theta(0) = 0.
    foreground = (directions @ e_sep) > 0.0
    transmission = np.exp(-kappa_absorber * ell_absorber * foreground[None, :])
    emergent = 1.0 - np.exp(-kappa_e * ell_emitter)
    return S_e * emergent * transmission


def ray_quantities(position, direction, kappa_absorber=None):
    """Chords, transmission, and f for one point and one direction."""
    if kappa_absorber is None:
        kappa_absorber = kappa_a
    position = np.asarray(position, dtype=np.float64)
    direction = np.asarray(direction, dtype=np.float64)
    direction = direction / np.linalg.norm(direction)
    ell_emitter = float(clipped_chord(position, direction, center_e, R_e))
    ell_absorber = float(clipped_chord(position, direction, center_a, R_a))
    foreground = bool(np.dot(direction, e_sep) > 0.0)
    optical_absorber = kappa_absorber * ell_absorber if foreground else 0.0
    transmission = float(np.exp(-optical_absorber))
    emergent = float(1.0 - np.exp(-kappa_e * ell_emitter))
    return {
        "ell_e": ell_emitter,
        "ell_a": ell_absorber,
        "foreground": foreground,
        "transmission": transmission,
        "emergent": emergent,
        "f": S_e * emergent * transmission,
    }


def separation_coordinates(position):
    """Axial coordinate from the emitter center, and cylindrical radius."""
    position = np.asarray(position, dtype=np.float64)
    relative = position - center_e
    axial = np.sum(relative * e_sep, axis=-1)
    transverse = relative - axial[..., None] * e_sep
    cylindrical_radius = np.linalg.norm(transverse, axis=-1)
    return axial, cylindrical_radius


def shadow_surfaces(axial):
    """Umbra, antumbra, and penumbra cylindrical radii at axial coordinate `axial`."""
    axial = np.asarray(axial, dtype=np.float64)
    varpi_umbra = (x_external - axial) * np.tan(beta_u)
    varpi_antumbra = (axial - x_external) * np.tan(beta_u)
    varpi_penumbra = (axial - x_internal) * np.tan(beta_p)
    return varpi_umbra, varpi_antumbra, varpi_penumbra


def shadow_region(position):
    """Opaque-limit region label. Finite-kappa f is still the chord formula."""
    position = np.asarray(position, dtype=np.float64)
    if np.linalg.norm(position - center_a) <= R_a:
        return "inside absorber"
    if np.linalg.norm(position - center_e) <= R_e:
        return "inside emitter"
    axial, cylindrical_radius = separation_coordinates(position)
    varpi_umbra, varpi_antumbra, varpi_penumbra = shadow_surfaces(float(axial))
    axial_umbra_start = d + R_a * np.sin(beta_u)
    axial_shadow_start = d - R_a * np.sin(beta_p)
    # The common-tangent surface is the boundary. A float residual of
    # ~1e-10 cm must not drop that surface into the interior label.
    on_umbra = abs(cylindrical_radius - varpi_umbra) <= 1e-6 * max(varpi_umbra, 1.0)
    on_antumbra = abs(cylindrical_radius - varpi_antumbra) <= 1e-6 * max(abs(varpi_antumbra), 1.0)
    on_penumbra = abs(cylindrical_radius - varpi_penumbra) <= 1e-6 * max(abs(varpi_penumbra), 1.0)
    if axial_umbra_start <= axial <= x_external and on_umbra:
        return "umbra boundary"
    if axial > x_external and on_antumbra:
        return "antumbra boundary"
    if axial >= axial_shadow_start and on_penumbra:
        return "penumbra boundary"
    if axial_umbra_start <= axial <= x_external and cylindrical_radius < varpi_umbra:
        return "umbra"
    if axial > x_external and cylindrical_radius < varpi_antumbra:
        return "antumbra"
    if axial >= axial_shadow_start and cylindrical_radius < varpi_penumbra:
        return "penumbra"
    return "unobstructed"


def cell_center(i, j, k):
    """Center of cell (i, j, k). Indices are zero-based."""
    return np.array(
        [(i + 0.5) * dx, (j + 0.5) * dy, (k + 0.5) * dz],
        dtype=np.float64,
    )


def cell_index(position):
    """Zero-based index of the cell containing `position`."""
    position = np.asarray(position, dtype=np.float64)
    return (
        int(np.floor(position[0] / dx)),
        int(np.floor(position[1] / dy)),
        int(np.floor(position[2] / dz)),
    )


def f_at_cell(i, j, k):
    """f[i, j, k, a] at the cell center, for every quadrature direction."""
    values = f_on_points(cell_center(i, j, k), phat)
    return values[0]


def moments_from_f(distribution, directions=None, phase_volume=None):
    """n and F from occupation samples on a common direction set.

    `distribution` has the direction axis last. `phase_volume` is the
    unscaled per-direction Vphase. The sums are

        n = sum_a f_a * Vphase_a / (2*pi*hbar*c)^3
        F = sum_a f_a * n_hat_a * Vphase_a / (2*pi*hbar*c)^3

    Units are 1/cm^3. There is no factor of c.
    """
    if directions is None:
        directions = phat
    if phase_volume is None:
        phase_volume = vphase_a
    distribution = np.asarray(distribution, dtype=np.float64)
    directions = np.asarray(directions, dtype=np.float64)
    phase_volume = np.asarray(phase_volume, dtype=np.float64)
    prefactor = phase_volume / (2.0 * np.pi * hbar * c) ** 3
    number_density = np.sum(distribution * prefactor, axis=-1)
    number_flux = np.einsum(
        "...a,aj,a->...j",
        distribution,
        directions,
        prefactor,
        optimize=True,
    )
    return number_density, number_flux


def moments_at_cell(i, j, k):
    """n[i, j, k] and F[i, j, k, :] from f at that cell center."""
    number_density, number_flux = moments_from_f(f_at_cell(i, j, k))
    return float(number_density), number_flux


def evaluate_chunk(i0, i1, j0=0, j1=None, k0=0, k1=None, store_f=True, max_f_bytes=2 * 1024**3):
    """Evaluate a rectangular block of cells, on cell centers.

    Index ranges are half-open. Returns
        f_chunk with shape (i1-i0, j1-j0, k1-k0, n_directions) or None,
        n_chunk  with shape (i1-i0, j1-j0, k1-k0),
        F_chunk  with shape (i1-i0, j1-j0, k1-k0, 3).
    Refuses to allocate `f` above `max_f_bytes`. The full grid at this
    resolution is about 8.82 GiB, so request slabs, or one cell via
    f_at_cell.
    """
    if j1 is None:
        j1 = ncell[1]
    if k1 is None:
        k1 = ncell[2]
    if not (
        0 <= i0 < i1 <= ncell[0]
        and 0 <= j0 < j1 <= ncell[1]
        and 0 <= k0 < k1 <= ncell[2]
    ):
        raise IndexError(
            f"chunk [{i0}:{i1}, {j0}:{j1}, {k0}:{k1}] is outside ncell={ncell}"
        )
    n_i = i1 - i0
    n_j = j1 - j0
    n_k = k1 - k0
    f_bytes = n_i * n_j * n_k * n_directions * np.dtype(np.float64).itemsize
    if store_f and f_bytes > max_f_bytes:
        raise MemoryError(
            f"f chunk would be {f_bytes / 1024**3:.2f} GiB "
            f"(full grid {full_f_bytes / 1024**3:.2f} GiB). "
            "Use a thinner slab, store_f=False, or f_at_cell(i, j, k)."
        )
    x_centers = (np.arange(i0, i1) + 0.5) * dx
    y_centers = (np.arange(j0, j1) + 0.5) * dy
    z_centers = (np.arange(k0, k1) + 0.5) * dz
    xx, yy, zz = np.meshgrid(x_centers, y_centers, z_centers, indexing="ij")
    points = np.stack((xx, yy, zz), axis=-1).reshape(-1, 3)
    distribution = f_on_points(points, phat)
    number_density, number_flux = moments_from_f(distribution)
    number_density = number_density.reshape(n_i, n_j, n_k)
    number_flux = number_flux.reshape(n_i, n_j, n_k, 3)
    if not store_f:
        return None, number_density, number_flux
    return distribution.reshape(n_i, n_j, n_k, n_directions), number_density, number_flux


In [ ]:
# Cell near the umbral surface, on a downstream cut halfway from the
# external tangency circle to the tip. Retrieval is at the cell center.
axial_umbra_start = d + R_a * np.sin(beta_u)
axial_cut = 0.5 * (axial_umbra_start + x_external)
varpi_umbra, varpi_antumbra, varpi_penumbra = shadow_surfaces(axial_cut)
position_cut = center_e + axial_cut * e_sep + varpi_umbra * np.array([0.0, 1.0, 0.0])
i_sel, j_sel, k_sel = cell_index(position_cut)
# Keep the demonstration inside the mesh.
i_sel = int(np.clip(i_sel, 0, ncell[0] - 1))
j_sel = int(np.clip(j_sel, 0, ncell[1] - 1))
k_sel = int(np.clip(k_sel, 0, ncell[2] - 1))

center_sel = cell_center(i_sel, j_sel, k_sel)
f_ijk = f_at_cell(i_sel, j_sel, k_sel)  # shape (n_directions,)
n_ijk, F_ijk = moments_at_cell(i_sel, j_sel, k_sel)

print(f"selected cell (i, j, k) = {(i_sel, j_sel, k_sel)}")
print(f"cell center = {center_sel}")
print(f"region label at the cell center = {shadow_region(center_sel)}")
print(
    f"at this axial cut, varpi_umbra = {float(varpi_umbra):.6g} cm, "
    f"varpi_penumbra = {float(varpi_penumbra):.6g} cm"
)
print(f"f[i,j,k,a] shape = {f_ijk.shape}, min = {f_ijk.min():.3e}, max = {f_ijk.max():.3e}")
print(f"n[i,j,k] = {n_ijk:.6e} cm^-3")
print(f"F[i,j,k,:] = {F_ijk} cm^-3")
print("one x-slab would be "
      f"{ncell[1] * ncell[2] * n_directions * 8 / 1024**2:.1f} MiB of f; "
      "call evaluate_chunk(i, i+1) for that slab")


In [ ]:
# Validation. Constructed rays test the chord formula. The theta-phi mesh
# is then used for the discrete moments.

def expect_close(name, got, reference, rtol=1e-12, atol=0.0):
    got = np.asarray(got, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    if not np.allclose(got, reference, rtol=rtol, atol=atol):
        raise AssertionError(f"{name}: got {got}, reference {reference}")
    print(f"pass  {name}")


# --- Chord at the emitter center is the radius, for every direction. ---
ell_center = clipped_chord(center_e, phat, center_e, R_e)
expect_close("interior center chord", ell_center, np.full(n_directions, R_e), atol=1e-6)

# Interior formula ell = b + sqrt(R^2 - D^2 + b^2), b = r · n.
interior_point = center_e + np.array([0.3 * R_e, -0.2 * R_e, 0.1 * R_e])
interior_direction = phat[0]
relative = interior_point - center_e
projection = float(np.dot(relative, interior_direction))
distance_squared = float(np.dot(relative, relative))
delta_interior = R_e**2 - distance_squared + projection**2
ell_interior_ref = projection + np.sqrt(delta_interior)
expect_close(
    "interior off-center chord",
    clipped_chord(interior_point, interior_direction, center_e, R_e),
    ell_interior_ref,
    atol=1e-6,
)

# --- Unobstructed ray: on axis, upstream, traveling toward -e_sep. ---
# The backward ray goes through a diameter of the emitter and then the
# absorber, but the absorber is not foreground (n · e_sep < 0).
position_upstream = center_e - 1.2 * R_e * e_sep
direction_upstream = -e_sep
upstream = ray_quantities(position_upstream, direction_upstream)
f_upstream_ref = S_e * (1.0 - np.exp(-kappa_e * 2.0 * R_e))
if upstream["foreground"]:
    raise AssertionError("upstream on-axis ray was marked foreground")
if upstream["ell_a"] <= 0.0:
    raise AssertionError("upstream ray should still intersect the far absorber")
expect_close("unobstructed emitter chord", upstream["ell_e"], 2.0 * R_e, atol=1e-4)
expect_close("unobstructed transmission", upstream["transmission"], 1.0)
expect_close("unobstructed f", upstream["f"], f_upstream_ref, rtol=1e-12)
print(
    f"      unobstructed ray: ell_e = {upstream['ell_e']:.6g} cm, "
    f"ell_a (not applied) = {upstream['ell_a']:.6g} cm, f = {upstream['f']:.6e}"
)

# --- Ray through the absorber: on axis, downstream, traveling toward +e_sep. ---
position_downstream = center_e + (d + 1.2 * R_a) * e_sep
direction_downstream = e_sep.copy()
downstream = ray_quantities(position_downstream, direction_downstream)
f_downstream_ref = f_upstream_ref * np.exp(-kappa_a * 2.0 * R_a)
if not downstream["foreground"]:
    raise AssertionError("downstream on-axis ray was not marked foreground")
expect_close("absorbed emitter chord", downstream["ell_e"], 2.0 * R_e, atol=1e-4)
expect_close("absorbed absorber chord", downstream["ell_a"], 2.0 * R_a, atol=1e-4)
expect_close("absorbed f", downstream["f"], f_downstream_ref, rtol=1e-12)
print(
    f"      absorbed ray: ell_e = {downstream['ell_e']:.6g} cm, "
    f"ell_a = {downstream['ell_a']:.6g} cm, "
    f"T_a = {downstream['transmission']:.6e}, f = {downstream['f']:.6e}"
)

# Same downstream point, nearest mesh direction to +e_sep, against the chord formula.
direction_index = int(np.argmax(phat @ e_sep))
emu_ray = ray_quantities(position_downstream, phat[direction_index])
f_emu = float(f_on_points(position_downstream, phat[direction_index])[0, 0])
expect_close("mesh-direction f matches chord formula", f_emu, emu_ray["f"], rtol=1e-12)
print(
    f"      nearest mesh direction to +e_sep has n·e_sep = {phat[direction_index] @ e_sep:.6f}, "
    f"f = {f_emu:.6e}"
)

# --- Emitter limb: f vanishes on and outside the limb and falls onto it. ---
# Observer sits on the +y side of the emitter. n_x = 0, so the absorber
# is never applied and the test isolates the emitter chord.
# Impact parameter = D * n_z. At p = R the geometric chord is zero; in
# float64 Delta underflows to a fraction of a centimeter, so f is compared
# with a tolerance rather than with exact zero.
limb_distance = 2.0 * R_e
limb_origin = center_e + np.array([0.0, limb_distance, 0.0])

def limb_direction(impact_parameter):
    sine = impact_parameter / limb_distance
    return np.array([0.0, np.sqrt(1.0 - sine * sine), sine])

f_face = ray_quantities(limb_origin, np.array([0.0, 1.0, 0.0]))["f"]
ray_near_limb = ray_quantities(limb_origin, limb_direction(0.99 * R_e))
ray_on_limb = ray_quantities(limb_origin, limb_direction(R_e))
ray_miss = ray_quantities(limb_origin, limb_direction(1.05 * R_e))
expect_close("face-on limb-test f", f_face, f_upstream_ref, rtol=1e-12)
expect_close("ray missing the emitter gives f = 0", ray_miss["f"], 0.0, atol=0.0)
if ray_on_limb["ell_e"] > 1.0:
    raise AssertionError(f"grazing chord is not numerically small: ell_e={ray_on_limb['ell_e']}")
if not (ray_on_limb["f"] < 1e-6 * S_e):
    raise AssertionError(f"grazing occupation is not negligible: f={ray_on_limb['f']}")
print("pass  grazing emitter chord is numerically negligible")
if not (0.0 < ray_near_limb["f"] < f_face):
    raise AssertionError("occupation does not rise from the limb toward the face")
print("pass  occupation rises from the limb toward the disk center")
print(
    f"      limb continuity: f(face) = {f_face:.6e}, "
    f"f(0.99 R) = {ray_near_limb['f']:.6e}, "
    f"f(limb) = {ray_on_limb['f']:.3e} (ell_e = {ray_on_limb['ell_e']:.3e} cm), "
    f"f(miss) = {ray_miss['f']:.3e}"
)

# --- Shadow boundary: external common tangent. ---
# A point on the downstream external tangent, between the absorber's
# tangency circle and the umbral tip. The parallel ray grazes both spheres.
e_perp = np.array([0.0, 1.0, 0.0])
axial_touch = d + R_a * np.sin(beta_u)
axial_boundary = 0.5 * (axial_touch + x_external)
varpi_boundary = float((x_external - axial_boundary) * np.tan(beta_u))
position_tangent = center_e + axial_boundary * e_sep + varpi_boundary * e_perp
direction_tangent = np.cos(beta_u) * e_sep - np.sin(beta_u) * e_perp
tangent = ray_quantities(position_tangent, direction_tangent)
# 10 cm toward the axis is a secant; 10 cm outward is a miss.
position_inward = position_tangent - 10.0 * e_perp
position_outward = position_tangent + 10.0 * e_perp
inward = ray_quantities(position_inward, direction_tangent)
outward = ray_quantities(position_outward, direction_tangent)
if tangent["ell_e"] > 1e-3 or tangent["ell_a"] > 1e-3:
    raise AssertionError(
        f"tangent chords are not numerically zero: ell_e={tangent['ell_e']}, ell_a={tangent['ell_a']}"
    )
if tangent["f"] > 1e-18:
    raise AssertionError(f"tangent occupation is not ~0: f={tangent['f']}")
if not (inward["ell_e"] > tangent["ell_e"] and inward["f"] > tangent["f"]):
    raise AssertionError("moving inside the shadow edge did not open a chord")
if outward["ell_e"] > 1e-3 or outward["f"] > 1e-18:
    raise AssertionError("moving outside the shadow edge still intersects a sphere")
print(
    f"      shadow boundary ({shadow_region(position_tangent)}): "
    f"ell_e = {tangent['ell_e']:.3e} cm, ell_a = {tangent['ell_a']:.3e} cm, "
    f"f = {tangent['f']:.3e}"
)
print(
    f"      10 cm inside the edge: ell_e = {inward['ell_e']:.6g} cm, "
    f"ell_a = {inward['ell_a']:.6g} cm, f = {inward['f']:.6e}"
)
print(
    f"      10 cm outside the edge: ell_e = {outward['ell_e']:.3e} cm, "
    f"f = {outward['f']:.3e}"
)

# Cell whose center is nearest this boundary point.
i_b, j_b, k_b = cell_index(position_tangent)
center_boundary = cell_center(i_b, j_b, k_b)
print(
    f"      nearest cell {(i_b, j_b, k_b)} is {shadow_region(center_boundary)}, "
    f"center = {center_boundary}"
)

# --- Null test: in the vacuum gap, every mesh direction matches kappa_a = 0. ---
axial_gap = 0.5 * (R_e + d - R_a)
position_gap = center_e + axial_gap * e_sep
if shadow_region(position_gap) != "unobstructed":
    raise AssertionError(f"gap point labeled {shadow_region(position_gap)}")
f_gap = f_on_points(position_gap, phat)[0]
f_gap_isolated = f_on_points(position_gap, phat, kappa_absorber=0.0)[0]
expect_close("gap equals isolated emitter", f_gap, f_gap_isolated, rtol=0.0, atol=0.0)

# Inside the emitter, the absorber does not change f.
f_inside = f_on_points(center_e, phat)[0]
f_inside_isolated = f_on_points(center_e, phat, kappa_absorber=0.0)[0]
expect_close("emitter interior equals isolated sphere", f_inside, f_inside_isolated, atol=0.0)
f_inside_ref = S_e * (1.0 - np.exp(-kappa_e * R_e))
expect_close("emitter-center occupation", f_inside, np.full(n_directions, f_inside_ref), rtol=1e-12)

# --- n and F are the direct weighted sum of that cell's f. ---
f_cell = f_at_cell(i_b, j_b, k_b)
prefactor = vphase_a / (2.0 * np.pi * hbar * c) ** 3
n_direct = float(np.sum(prefactor * f_cell))
F_direct = np.sum(prefactor[:, None] * f_cell[:, None] * phat, axis=0)
n_api, F_api = moments_at_cell(i_b, j_b, k_b)
expect_close("n matches direct sum of f", n_api, n_direct, rtol=1e-12)
expect_close("F matches direct sum of f", F_api, F_direct, rtol=1e-12, atol=1e-14)
print(f"      boundary cell n = {n_api:.6e} cm^-3, F = {F_api}")

# One-plane chunk agrees with the single-cell retrieval.
f_plane, n_plane, F_plane = evaluate_chunk(i_b, i_b + 1)
expect_close("chunk f matches f_at_cell", f_plane[0, j_b, k_b], f_cell, rtol=0.0, atol=0.0)
expect_close("chunk n matches moments_at_cell", n_plane[0, j_b, k_b], n_api, rtol=1e-12)
expect_close("chunk F matches moments_at_cell", F_plane[0, j_b, k_b], F_api, rtol=1e-12, atol=1e-14)

# The full distribution is not allocated. A too-large request must fail cleanly.
try:
    evaluate_chunk(0, ncell[0])
except MemoryError as error:
    print(f"pass  full-grid f request refused: {error}")
else:
    raise AssertionError("full-grid f allocation was not refused")

print("all validation checks passed")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Number density on the plane z index = 32, all x and y cell centers.
k_slice = 32
_, n_chunk, _ = evaluate_chunk(
    0, ncell[0], 0, ncell[1], k_slice, k_slice + 1, store_f=False
)
n_xy = n_chunk[:, :, 0]

x_centers = (np.arange(ncell[0]) + 0.5) * dx
y_centers = (np.arange(ncell[1]) + 0.5) * dy
z_center = (k_slice + 0.5) * dz

n_plot = np.ma.masked_less_equal(n_xy, 0.0)
positive = n_xy[n_xy > 0.0]
vmin = float(np.min(positive))
vmax = float(np.max(positive))

fig, ax = plt.subplots(figsize=(11.5, 4.4))
mesh = ax.pcolormesh(
    x_centers,
    y_centers,
    n_plot.T,
    shading="nearest",
    cmap="inferno",
    norm=LogNorm(vmin=vmin, vmax=vmax),
)
ax.set_aspect("equal")
ax.set_xlabel(r"$x$ (cm)")
ax.set_ylabel(r"$y$ (cm)")
ax.set_title(
    rf"Analytic number density at $z$ index {k_slice} "
    rf"($z = {z_center:.6g}$ cm)"
)
colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label(r"number density $n$ (cm$^{-3}$)")

# Cross sections of the two spheres in this midplane.
angle = np.linspace(0.0, 2.0 * np.pi, 512)
ax.plot(
    center_e[0] + R_e * np.cos(angle),
    center_e[1] + R_e * np.sin(angle),
    color="cyan",
    lw=0.9,
    label="emitter",
)
ax.plot(
    center_a[0] + R_a * np.cos(angle),
    center_a[1] + R_a * np.sin(angle),
    color="white",
    lw=0.9,
    label="absorber",
)
ax.legend(loc="upper right", frameon=True)
fig.tight_layout()
plt.show()

print(
    f"k = {k_slice}, n shape = {n_xy.shape}, "
    f"min = {n_xy.min():.6e}, max = {n_xy.max():.6e} cm^-3"
)
print(f"cells with n = 0: {np.count_nonzero(n_xy <= 0)} / {n_xy.size}")


In [ ]:
def occupation_number(x, y, z, phat):
    """Occupation number f at (x, y, z) for a unit momentum direction.

    x, y, z are in cm. phat is a unit vector, shape (3,), or several
    directions of shape (A, 3). A single direction returns a float.
    """
    position = np.array([x, y, z], dtype=np.float64)
    direction = np.asarray(phat, dtype=np.float64)
    values = f_on_points(position, direction)[0]
    if direction.ndim == 1:
        return float(values[0])
    return values


# Direction from the emitter center toward a downstream on-axis point.
x_demo = center_e[0] + (d + 1.2 * R_a)
y_demo = center_e[1]
z_demo = center_e[2]
f_demo = occupation_number(x_demo, y_demo, z_demo, e_sep)
print(f"f({x_demo:.6g}, {y_demo:.6g}, {z_demo:.6g}, e_sep) = {f_demo:.6e}")


In [ ]:
def number_density_and_flux(x, y, z):
    """Number density n and number flux F at (x, y, z).

    Both are in cm^-3. F is the EMU flux, the number flux divided by c.
    The sum uses the notebook theta-phi mesh and its solid-angle weights.
    """
    position = np.array([x, y, z], dtype=np.float64)
    distribution = f_on_points(position, phat)[0]
    number_density, number_flux = moments_from_f(distribution)
    return float(number_density), np.asarray(number_flux, dtype=np.float64)


n_demo, F_demo = number_density_and_flux(x_demo, y_demo, z_demo)
print(f"n = {n_demo:.6e} cm^-3")
print(f"F = {F_demo} cm^-3")
